In [1]:
%load_ext autoreload
%autoreload 2

In [62]:
import logging
import sys

from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts import (
    generate_dataset,
    optuna_search,
    predictions,
    training,
)
from experiments.plotting import _plot_SRE_distribution, view_correlation

from src.utils import configure_logger

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

In [63]:
logger = logging.getLogger(__name__)
configure_logger(logging.INFO, logging.INFO)

True

In [66]:
backend = "pennylane"
target = "SRE"
method = "fwht"
use_dask = True

output_dir = "outputs/data/SRE_datasets"
n_bins = 50
families = "random"
n_seeds_train = 50
n_seeds_pred = 10
qubit_min, qubit_max = 4, 10
layer_min, layer_max = 2, 100
step = 2
target_qubits = "4,6,8"
max_config = None
dask_workers = 4
dask_memory = "4GiB"

In [67]:
generate_dataset(
    backend=backend,
    target=target,
    method=method,
    output_dir=output_dir,
    n_bins_option=n_bins,
    families=families,
    n_seeds_option=n_seeds_train,
    prediction_n_seeds_option=n_seeds_pred,
    qubits_min=qubit_min,
    qubits_max=qubit_max,
    layers_min=layer_min,
    layers_max=layer_max,
    qubits_step=step,
    layers_step=step,
    target_qubits=target_qubits,
    max_configs=max_config,
    use_dask=use_dask,
    dask_n_workers=dask_workers,
    dask_memory_per_worker=dask_memory,
    block_size = 10,
)

2026-06-01 17:17:15,540 - GNN.dataset_builder - INFO - Processing family: random
2026-06-01 17:17:15,607 - GNN.dataset_builder - INFO - Generated 24 shards for random
2026-06-01 17:17:15,610 - parallel.dask - INFO - Creating local Dask cluster with 4 workers, 1 threads per worker.
2026-06-01 17:17:18,516 - parallel.dask - INFO - Dask client created successfully.
2026-06-01 17:17:18,519 - parallel.dask - INFO - Client dashboard available at: http://127.0.0.1:8787/status
Parallel dataset generation: 100%|██████████| 24/24 [30:48<00:00, 77.04s/it]  
2026-06-01 17:48:12,119 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-06-01 17:48:12,186 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-06-01 17:48:12,189 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-06-01 17:48:12,192 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-06

In [68]:
import ast

def to_scalar(x):
    # Already numeric
    if isinstance(x, (int, float)):
        return x

    # torch / numpy scalar
    if hasattr(x, "item"):
        return x.item()

    # Strings
    if isinstance(x, str):
        x = x.strip()

        # Handle tensor(...) by stripping wrapper FIRST
        if x.startswith("tensor(") and x.endswith(")"):
            x = x[len("tensor("):-1].strip()

        try:
            val = ast.literal_eval(x)
        except Exception:
            # fallback: plain float string
            return float(x)

        # If it's a list/tuple like [10]
        if isinstance(val, (list, tuple)):
            if len(val) == 1:
                return float(val[0])
            raise ValueError(f"Unexpected list length: {val}")

        return float(val)

    raise ValueError(f"Unsupported type: {type(x)}")

In [69]:
def run(
    model_type,
    epochs,
    lr,
    loss_type,
    batch_size,
    training_mode,
    family,
    target,
    target_variant,
    model_hparams,
    train_hparams,
    training_data_dir,
    model_save_path,
    split,
    plot_qubits=10,
    plot_layers=80,
):
    training(
        epochs=epochs,
        lr=lr,
        loss_type=loss_type,
        batch_size=batch_size,
        training_mode=training_mode,
        family=family,
        target=target,
        target_variant=target_variant,
        model_type=model_type,
        model_hparams=model_hparams,
        train_hparams=train_hparams,
        training_data_dir=training_data_dir,
        split=split,
        model_save_path=model_save_path,
        show_progress=True,
        show_val_progress=False,
        log_every_n_batches=10,
        heartbeat_secs=60.0,
        epoch_time_warning_secs=600.0,
    )
    training_scope = "family" if training_mode == "per_family" else "global"
    predictions(
        model_path=model_save_path,
        model_kind=model_type,
        training_scope=training_scope,
        loss_type=loss_type,
        model_family=family,
        dataset_root=training_data_dir,
        dataset_family=family,
        batch_size=batch_size,
        global_feature_variant="binned",
        node_feature_backend_variant=None,
        plot_n_layers=plot_layers,
        plot_n_qubits=plot_qubits,
        split_by_family=True,
        show_progress=True,
    )

    df = pd.read_csv(f"../outputs/predictions/{training_scope}/{model_type}_predictions_{family}.csv")
    cols_to_fix = ["n_qubits", "n_layers", "seed"]

    for col in cols_to_fix:
        df[col] = df[col].apply(to_scalar).astype(int)
    view_correlation(
        df,
        nq=plot_qubits,
        nl=plot_layers,
        col_x="target",
        col_y="prediction",
    )
    df = df[(df["n_qubits"] == plot_qubits) & (df["n_layers"] == plot_layers)]
    plt.figure(figsize=(8, 6))
    plt.scatter(df["target"], df["prediction"], alpha=0.7)
    plt.plot([df["target"].min(), df["target"].max()], [df["target"].min(), df["target"].max()], "r--")  # y=x line
    plt.xlabel("True SRE")
    plt.ylabel("Predicted SRE")
    plt.title("True vs Predicted SRE for Clifford Family (GNN Model)")

In [72]:
model_type="gnn"
epochs = 5
lr = 0.0018760928244666698
loss_type = "huber"   # "mse" | "huber"
batch_size = 16
training_mode = "per_family"  # "global" | "per_family"
family = "random"  # required if training_mode == "per_family"
target = "sre"
data_dir = "../outputs/data/SRE_datasets"
model_save_path = f"../outputs/models/{family}_model_{model_type}_{training_mode}_2.pt"
show_progress=True
show_val_progress=False
log_every_n_batches=10
heartbeat_secs=60.0
epoch_time_warning_secs=600.0
training_scope = "family" if training_mode == "per_family" else "global"
plot_qubits = 6
plot_layers = 100
target_variant = "sre_density"  # "sre" | "sre_density" | "log_sre" | "sqrt_sre"
split = "target"  # "target" | "family"


model_hparams = {
    "gnn_hidden": 32,
    "gnn_heads": 4,
    "global_hidden": 128,
    "reg_hidden": 128,
    "num_layers": 3,
    "dropout_rate": 0.13173830279748305,
}

train_hparams = {
    "weight_decay": 0.0003324725858640221,
    "grad_clip": 1.0289214665544766,
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.0,
}


run(
    model_type=model_type,
    epochs=epochs,
    lr=lr,
    loss_type=loss_type,
    batch_size=batch_size,
    training_mode=training_mode,
    family=family,
    target=target,
    target_variant=target_variant,
    model_hparams=model_hparams,
    train_hparams=train_hparams,
    training_data_dir=data_dir,
    split=split,
    model_save_path=model_save_path,
)

2026-06-01 17:56:59,480 - GNN.training.runners - INFO - Starting training | model_type=gnn | training_mode=per_family | family=random | loss_type=huber
2026-06-01 17:56:59,483 - GNN.training.runners - INFO - Training configuration done.
2026-06-01 17:56:59,485 - GNN.training.runners - INFO - Collecting data paths...
2026-06-01 17:56:59,490 - GNN.training.runners - INFO - Found 1 data paths.
2026-06-01 17:56:59,494 - GNN.training.runners - INFO - Data paths collected.
2026-06-01 17:56:59,496 - GNN.training.runners - INFO - Building loaders and model for model_type=gnn...
2026-06-01 17:57:57,038 - GNN.training.runners - INFO - Loaders and model built.
2026-06-01 17:57:57,041 - GNN.training.runners - INFO - Starting training...
2026-06-01 17:57:57,099 - GNN.training.train - INFO - -------- EPOCH 001 --------


KeyboardInterrupt: 